
# Mock Data + Multi-Class Dirichlet GP Classification

In [1]:
import sys
from pathlib import Path

project_root = Path(r"c:\Users\floki\PycharmProjects\dash-azure-prototype")
src_root = project_root / "src"
for p in (project_root, src_root):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
artifact_dir = project_root / "ml" / "models" / "artifacts" / "demo_gpc"
artifact_dir.mkdir(parents=True, exist_ok=True)

In [2]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_wine, load_digits

In [3]:
from ml.scripts.trainers import GPCTrainer
from ml.scripts.log import init_ml_logger, log_classification_summary
from worker.models import GPC
from worker.models.specs import ModelConfig, PreprocessConfig, AuxilaryData
from worker.models.io_utils import ArtifactIO
from worker.torch_utils import get_default_device

In [4]:
# load demo data for testing
dataset = load_digits(as_frame=True)

X = dataset.data.copy()

# 10 binary target columns: one per digit class
y = pd.get_dummies(dataset.target, prefix="target").astype(int)

In [5]:
print(f"n observations {X.shape[0]}")

n observations 1797


In [6]:
pd.concat((X, y), axis=1).head()

,pixel_0_0,pixel_0_1,pixel_0_2,pixel_0_3,pixel_0_4,pixel_0_5,pixel_0_6,pixel_0_7,pixel_1_0,pixel_1_1,...,target_0,target_1,target_2,target_3,target_4,target_5,target_6,target_7,target_8,target_9
0,0.0,0.0,5.0,13.0,9.0,1.0,0.0,0.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,0
1,0.0,0.0,0.0,12.0,13.0,5.0,0.0,0.0,0.0,0.0,...,0,1,0,0,0,0,0,0,0,0
2,0.0,0.0,0.0,4.0,15.0,12.0,0.0,0.0,0.0,0.0,...,0,0,1,0,0,0,0,0,0,0
3,0.0,0.0,7.0,15.0,13.0,1.0,0.0,0.0,0.0,8.0,...,0,0,0,1,0,0,0,0,0,0
4,0.0,0.0,0.0,1.0,11.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,1,0,0,0,0,0


In [7]:
# train test split
feature_cols = [f"X{i}" for i in range(X.shape[1])]
target_cols = [f"Y{i}" for i in range(y.shape[1])]

X.columns = feature_cols
y.columns = target_cols

X = X[feature_cols].to_numpy()
y = y[target_cols].to_numpy()

train_x, test_x, train_y, test_y = train_test_split(
    X, y, test_size=0.33, random_state=1, shuffle=True, stratify=dataset.target
)

train_x_raw = train_x.copy()
test_x_raw = test_x.copy()
train_y_raw = train_y.copy()
test_y_raw = test_y.copy()

In [8]:
# feature scaling
scaler_x = StandardScaler()
train_x = scaler_x.fit_transform(train_x)
test_x = scaler_x.transform(test_x)

In [9]:
# setup model
prep = PreprocessConfig(scaler_x=scaler_x)
spec = ModelConfig(
    model_type="gpc",
    features=feature_cols,
    targets=target_cols,
    requires_aux=True,
)
aux = AuxilaryData(train_x=train_x, train_y=train_y)
model = GPC(spec, prep, aux)

In [10]:
# conduct training
device = get_default_device()
print(f"Running on {str(device)}")
trainer = GPCTrainer(model, device=device, lr=0.15)
metrics = trainer.train(train_x, train_y, epochs=150)

Running on cuda


Dirichlet GP training:   0%|          | 0/150 [00:00<?, ?it/s]

In [11]:
# check if storing / loading of artifacts works
ArtifactIO.save(artifact_dir, model=model, spec=spec, prep=prep, aux=aux)
loaded_model = ArtifactIO.load(artifact_dir, device=device)

In [12]:
# test predictions on test set
preds = loaded_model.predict(test_x_raw, device=device)
pred_cls = preds.get('cls')
pred_prob = preds.get('prob')

preds_train = loaded_model.predict(train_x_raw, device=device)
pred_cls_train = preds_train.get('cls')
pred_prob_train = preds_train.get('prob')

In [13]:
logger, _ = init_ml_logger(artifact_dir, "training")
if logger is not None:
    logger.info("Train time (s): %.3f", trainer.duration)
    log_classification_summary(
        logger,
        y_true=train_y_raw,
        y_pred=pred_cls_train,
        y_score=pred_prob_train,
        target_cols=target_cols,
        phase="Train",
        average="binary",
    )
    log_classification_summary(
        logger,
        y_true=test_y_raw,
        y_pred=pred_cls,
        y_score=pred_prob,
        target_cols=target_cols,
        phase="Test",
        average="binary",
    )

2026-04-02 00:13:47,824 | INFO | Train time (s): 93.773
2026-04-02 00:13:47,827 | INFO | Train: n_obs=1203 n_targets=10 non_nan_targets=12030
2026-04-02 00:13:47,827 | INFO | Train classification metrics per target:
2026-04-02 00:13:47,835 | INFO |   Y0 | Acc=0.9992 BalAcc=0.9958 Precision=1.0000 Recall=0.9916 F1=0.9958 Support=nan
2026-04-02 00:13:47,838 | INFO |   Y1 | Acc=0.9975 BalAcc=0.9877 Precision=1.0000 Recall=0.9754 F1=0.9876 Support=nan
2026-04-02 00:13:47,842 | INFO |   Y2 | Acc=1.0000 BalAcc=1.0000 Precision=1.0000 Recall=1.0000 F1=1.0000 Support=nan
2026-04-02 00:13:47,848 | INFO |   Y3 | Acc=0.9933 BalAcc=0.9675 Precision=1.0000 Recall=0.9350 F1=0.9664 Support=nan
2026-04-02 00:13:47,850 | INFO |   Y4 | Acc=0.9967 BalAcc=0.9835 Precision=1.0000 Recall=0.9669 F1=0.9832 Support=nan
2026-04-02 00:13:47,855 | INFO |   Y5 | Acc=0.9967 BalAcc=0.9836 Precision=1.0000 Recall=0.9672 F1=0.9833 Support=nan
2026-04-02 00:13:47,858 | INFO |   Y6 | Acc=0.9983 BalAcc=0.9917 Precision=1

In [ ]:
# Prediction + metrics summary (unscaled)
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
)

reports_dir = artifact_dir / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

def _class_metrics_table(y_true, y_pred, y_score, target_cols, average="binary", zero_division=0.0):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_score = None if y_score is None else np.asarray(y_score, dtype=float)

    if y_true.ndim == 1:
        y_true = y_true.reshape(-1, 1)
    if y_pred.ndim == 1:
        y_pred = y_pred.reshape(-1, 1)
    if y_score is not None and y_score.ndim == 1:
        y_score = y_score.reshape(-1, 1)

    labels = target_cols or [f"y{i}" for i in range(y_true.shape[1])]
    rows = []
    for i, name in enumerate(labels):
        yt = y_true[:, i]
        yp = y_pred[:, i]

        mask = ~np.isnan(yt)
        if np.issubdtype(yp.dtype, np.floating):
            mask &= ~np.isnan(yp)

        yt = yt[mask]
        yp = yp[mask]
        ys = y_score[:, i][mask] if y_score is not None else None

        acc = np.nan
        bal_acc = np.nan
        prec = np.nan
        rec = np.nan
        f1 = np.nan
        support = np.nan
        roc_auc = np.nan
        avg_prec = np.nan

        if yt.size > 0:
            try:
                acc = accuracy_score(yt, yp)
            except Exception:
                pass
            try:
                bal_acc = balanced_accuracy_score(yt, yp)
            except Exception:
                pass
            try:
                p, r, f, s = precision_recall_fscore_support(
                    yt,
                    yp,
                    average=average,
                    zero_division=zero_division,
                )
                prec = p
                rec = r
                f1 = f
                support = float(s) if np.isscalar(s) else np.sum(s)
            except Exception:
                pass
            if ys is not None and np.unique(yt).size >= 2:
                try:
                    roc_auc = roc_auc_score(yt, ys)
                except Exception:
                    pass
                try:
                    avg_prec = average_precision_score(yt, ys)
                except Exception:
                    pass

        rows.append({
            "target": name,
            "accuracy": acc,
            "balanced_accuracy": bal_acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "support": support,
            "roc_auc": roc_auc,
            "avg_precision": avg_prec,
        })

    return pd.DataFrame(rows)

def _summary(x, y_true, y_pred, y_score, feature_cols, target_cols, phase):
    x = np.asarray(x)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_score = None if y_score is None else np.asarray(y_score, dtype=float)

    if x.ndim == 1:
        x = x.reshape(-1, 1)
    if y_true.ndim == 1:
        y_true = y_true.reshape(-1, 1)
    if y_pred.ndim == 1:
        y_pred = y_pred.reshape(-1, 1)
    if y_score is not None and y_score.ndim == 1:
        y_score = y_score.reshape(-1, 1)

    feature_cols = feature_cols or [f"X{i}" for i in range(x.shape[1])]
    target_cols = target_cols or [f"Y{i}" for i in range(y_true.shape[1])]

    data = {"row_index": np.arange(x.shape[0], dtype=int), "phase": [phase] * x.shape[0]}
    for i, name in enumerate(feature_cols):
        data[name] = x[:, i]
    for i, name in enumerate(target_cols):
        data[f"y_true_{name}"] = y_true[:, i]
        data[f"y_pred_{name}"] = y_pred[:, i]
        if y_score is not None:
            data[f"y_score_{name}"] = y_score[:, i]

    return pd.DataFrame(data)

metrics_train = _class_metrics_table(train_y_raw, pred_cls_train, pred_prob_train, target_cols)
metrics_train["phase"] = "Train"
metrics_test = _class_metrics_table(test_y_raw, pred_cls, pred_prob, target_cols)
metrics_test["phase"] = "Test"
metrics_df = pd.concat([metrics_train, metrics_test], ignore_index=True)
metrics_df.to_csv(reports_dir / "metrics_summary.csv", index=False)

summary_train = _summary(train_x_raw, train_y_raw, pred_cls_train, pred_prob_train, feature_cols, target_cols, "Train")
summary_test = _summary(test_x_raw, test_y_raw, pred_cls, pred_prob, feature_cols, target_cols, "Test")
summary_df = pd.concat([summary_train, summary_test], ignore_index=True)
summary_df.to_csv(reports_dir / "predictions_summary.csv", index=False)

pd.DataFrame().to_csv(reports_dir / "outliers.csv", index=False)

print(f"Summary rows: {len(summary_df)}")
display(metrics_df.head())
display(summary_df.head())
